In [ ]:
# Paths: set DATA_DIR / OUTPUT_DIR as environment variables, or edit the defaults below.
import os
DATA_DIR = os.environ.get("DATA_DIR", "../data")


In [1]:
%pip install osmnx geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.3 MB/s eta 0:00:00


In [2]:
import os
import gc
import pandas as pd
import numpy as np
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [3]:
pass  # Colab only: from google.colab import drive
pass  # Colab only: drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

BASE_DIR     = DATA_DIR + "/"
DATASETS_DIR = BASE_DIR + "datasets/"
MODELS_DIR   = BASE_DIR + "models/"
REGIONS_DIR  = BASE_DIR + "regions/"
NATIONAL_DIR = BASE_DIR + "national/"

for d in [DATASETS_DIR, MODELS_DIR, REGIONS_DIR, NATIONAL_DIR]:
    os.makedirs(d, exist_ok=True)

print("Directories ready")

Directories ready


In [5]:
required_files = [
    DATASETS_DIR + "collisions_dataset_clean.csv",
    DATASETS_DIR + "collisions_features.csv",
    MODELS_DIR   + "risk_model_v2.pkl",
    MODELS_DIR   + "le_time.pkl",
    MODELS_DIR   + "le_season.pkl",
    MODELS_DIR   + "le_surface.pkl",
]

all_present = True
for f in required_files:
    exists = os.path.exists(f)
    size   = f"{os.path.getsize(f)/1e6:.1f}MB" if exists else "MISSING"
    status = "" if exists else "***"
    print(f"  {status} {f.split('/')[-1]} — {size}")
    if not exists:
        all_present = False

print(f"\n{'All files present' if all_present else 'Missing files'}")


   collisions_dataset_clean.csv — 71.0MB
   collisions_features.csv — 95.8MB
   risk_model_v2.pkl — 0.6MB
  *** le_time.pkl — MISSING
  *** le_season.pkl — MISSING
  *** le_surface.pkl — MISSING

Missing files


In [6]:
import psutil
ram = psutil.virtual_memory()
print(f"\nRAM available: {ram.available/1e9:.1f}GB / {ram.total/1e9:.1f}GB total")


RAM available: 12.2GB / 13.6GB total


In [7]:
REGIONS = [
    # England
    {"id": "01",   "name": "Greater_London",        "query": "Greater London, England"},
    {"id": "02a",  "name": "Kent",                  "query": "Kent, England"},
    {"id": "02b",  "name": "Surrey",                "query": "Surrey, England"},
    {"id": "02c",  "name": "Hampshire",             "query": "Hampshire, England"},
    {"id": "02d",  "name": "Sussex",                "query": "Sussex, England"},
    {"id": "02e",  "name": "Berkshire",             "query": "Berkshire, England"},
    {"id": "02f",  "name": "Oxfordshire",           "query": "Oxfordshire, England"},
    {"id": "03a",  "name": "Devon",                 "query": "Devon, England"},
    {"id": "03b",  "name": "Cornwall",              "query": "Cornwall, England"},
    {"id": "03c",  "name": "Somerset",              "query": "Somerset, England"},
    {"id": "03d",  "name": "Gloucestershire",       "query": "Gloucestershire, England"},
    {"id": "03e",  "name": "Wiltshire",             "query": "Wiltshire, England"},
    {"id": "03f",  "name": "Dorset",                "query": "Dorset, England"},
    {"id": "04",   "name": "East_of_England",       "query": "East of England"},
    {"id": "05",   "name": "East_Midlands",         "query": "East Midlands, England"},
    {"id": "06",   "name": "West_Midlands",         "query": "West Midlands, England"},
    {"id": "07",   "name": "Yorkshire_and_Humber",  "query": "Yorkshire and the Humber, England"},
    {"id": "08a",  "name": "Lancashire",            "query": "Lancashire, England"},
    {"id": "08b",  "name": "Cumbria",               "query": "Cumbria, England"},
    {"id": "08c",  "name": "Cheshire",              "query": "Cheshire, England"},
    {"id": "09",   "name": "North_East_England",    "query": "North East, England"},
    {"id": "10",   "name": "Greater_Manchester",    "query": "Greater Manchester, UK", "done": True},
    {"id": "11",   "name": "Merseyside",            "query": "Merseyside, England"},
    {"id": "12",   "name": "West_Yorkshire",        "query": "West Yorkshire, England"},
    {"id": "13",   "name": "South_Yorkshire",       "query": "South Yorkshire, England"},
    {"id": "14",   "name": "Nottinghamshire",       "query": "Nottinghamshire, England"},
    # Scotland
    {"id": "15a",  "name": "Glasgow_City",          "query": "Glasgow City, Scotland"},
    {"id": "15b",  "name": "Lothian",               "query": "Lothian, Scotland"},
    {"id": "15c1", "name": "Argyll_and_Bute",       "query": "Argyll and Bute, Scotland"},
    {"id": "15c2", "name": "North_Ayrshire",        "query": "North Ayrshire, Scotland"},
    {"id": "15c3", "name": "South_Lanarkshire",     "query": "South Lanarkshire, Scotland"},
    {"id": "15c4", "name": "North_Lanarkshire",     "query": "North Lanarkshire, Scotland"},
    {"id": "16",   "name": "Highland_and_Islands",  "query": "Highland, Scotland"},
    {"id": "17",   "name": "East_Scotland",         "query": "Perth and Kinross, Scotland"},
    {"id": "18",   "name": "South_Scotland",        "query": "Dumfries and Galloway, Scotland"},
    # Wales
    {"id": "19",   "name": "North_Wales",           "query": "North Wales"},
    {"id": "20",   "name": "South_Wales",           "query": "South Wales"},
]

print(f"Total regions: {len(REGIONS)}")
print(f"Already done:  {sum(1 for r in REGIONS if r.get('done'))}")
print(f"To process:    {sum(1 for r in REGIONS if not r.get('done'))}")

Total regions: 37
Already done:  1
To process:    36


In [8]:
#Load collisions
print("\nLoading national collision data...")
collisions = pd.read_csv(
    DATASETS_DIR + "collisions_features.csv",
    dtype={'collision_index': str},
    low_memory=False
)
collisions['datetime'] = pd.to_datetime(collisions['datetime'])
print(f"{len(collisions):,} collisions loaded")
print(f"Date range: {collisions['datetime'].min()} → {collisions['datetime'].max()}")
print(f"RAM used: {collisions.memory_usage(deep=True).sum()/1e6:.1f}MB")


Loading national collision data...
503,407 collisions loaded
Date range: 2020-01-01 00:01:00 → 2024-12-31 23:30:00
RAM used: 398.0MB


In [9]:
#process_region function
import time

def process_region(region, collisions_df):
    """
    For a given region:
    1. Download OSM road network
    2. Clip accidents to region bounds
    3. Snap accidents to nearest road segment using BallTree
    4. Build grid-level historical risk features
    5. Save and return summary
    """
    rid     = region['id']
    name    = region['name']
    query   = region['query']
    outpath = REGIONS_DIR + f"region_{rid}_{name}.csv"

    if os.path.exists(outpath):
        existing = pd.read_csv(outpath)
        print(f" {name} — already done ({len(existing):,} grid cells)")
        return existing

    print(f"\n{'='*60}")
    print(f"  [{rid}] {name}")
    print(f"{'='*60}")
    t0 = time.time()

    try:
        print(f"  Downloading OSM network...")
        G     = ox.graph_from_place(query, network_type='drive', simplify=True)
        edges = ox.graph_to_gdfs(G, nodes=False).reset_index()
        edges = edges.to_crs('EPSG:4326')
        print(f"  Segments: {len(edges):,}")

        bounds = edges.total_bounds
        mask = (
            (collisions_df['longitude'] >= bounds[0]) &
            (collisions_df['longitude'] <= bounds[2]) &
            (collisions_df['latitude']  >= bounds[1]) &
            (collisions_df['latitude']  <= bounds[3])
        )
        rc = collisions_df[mask].copy()
        print(f"  Collisions in bounds: {len(rc):,}")

        if len(rc) < 10:
            print(f" Too few collisions — skipping")
            del G, edges; gc.collect()
            return None

        rc['grid_lat'] = rc['latitude'].round(2)
        rc['grid_lon'] = rc['longitude'].round(2)
        rc['grid_id']  = rc['grid_lat'].astype(str) + '_' + rc['grid_lon'].astype(str)

        print(f"  Snapping to road segments...")
        edges_proj = edges.to_crs('EPSG:27700')
        edges_proj['edge_id'] = range(len(edges_proj))

        midpoints   = edges_proj.geometry.interpolate(0.5, normalized=True)
        mid_latlon  = gpd.GeoSeries(midpoints, crs='EPSG:27700').to_crs('EPSG:4326')
        edge_coords = np.radians([[p.y, p.x] for p in mid_latlon])
        acc_coords  = np.radians(rc[['latitude','longitude']].values)

        tree = BallTree(edge_coords, metric='haversine')
        distances, indices = tree.query(acc_coords, k=1)
        dist_m = distances[:, 0] * 6371000

        rc['edge_id']     = edges_proj.iloc[indices[:, 0]]['edge_id'].values
        rc['snap_dist_m'] = dist_m
        rc = rc[rc['snap_dist_m'] <= 200]
        print(f"  After quality filter: {len(rc):,}")

        global_rate = rc['is_serious'].mean()
        prior       = 10

        grid = rc.groupby('grid_id').agg(
            total_accidents   = ('collision_index', 'count'),
            serious_accidents = ('is_serious', 'sum'),
            avg_speed_limit   = ('speed_limit', 'mean'),
            pct_dark          = ('is_dark', 'mean'),
            pct_wet           = ('is_wet_road', 'mean'),
            pct_rural         = ('is_rural_road', 'mean'),
            pct_high_speed    = ('is_high_speed', 'mean'),
            pct_at_junction   = ('at_junction', 'mean'),
            pct_bad_weather   = ('is_bad_weather', 'mean'),
            avg_lat           = ('latitude', 'mean'),
            avg_lon           = ('longitude', 'mean'),
        ).reset_index()

        grid['serious_rate']       = (
            (grid['serious_accidents'] + prior * global_rate) /
            (grid['total_accidents']   + prior)
        )
        grid['accidents_per_year'] = grid['total_accidents'] / 5
        grid['log_freq']           = np.log1p(grid['accidents_per_year'])

        scaler = MinMaxScaler()
        grid['hist_freq_score']     = scaler.fit_transform(grid[['log_freq']])
        grid['hist_severity_score'] = scaler.fit_transform(grid[['serious_rate']])
        grid['hist_speed_score']    = scaler.fit_transform(grid[['avg_speed_limit']])
        grid['base_risk_score']     = (
            0.60 * grid['hist_freq_score'] +
            0.30 * grid['hist_severity_score'] +
            0.10 * grid['hist_speed_score']
        )

        grid['region_id']   = rid
        grid['region_name'] = name

        grid.to_csv(outpath, index=False)
        elapsed = round(time.time() - t0, 1)
        print(f" Done in {elapsed}s — {len(grid):,} grid cells saved")

        del G, edges, edges_proj, rc, midpoints, mid_latlon
        gc.collect()
        return grid

    except Exception as e:
        print(f"*** Failed: {e}")
        gc.collect()
        return None

In [10]:
# Full national processing loop, skipping any already saved to Drive.

print("Starting national processing loop...\n")
results = []
failed  = []
pending = [r for r in REGIONS if not r.get('done')]

for i, region in enumerate(pending):
    print(f"\nProgress: {i+1}/{len(pending)}")
    result = process_region(region, collisions)
    if result is not None:
        results.append(result)
    else:
        failed.append(region['name'])

print(f"NATIONAL PROCESSING COMPLETE")
print(f"Regions processed: {len(results)}")
print(f"Regions failed:    {len(failed)}")
if failed:
    print(f"Failed: {failed}")

Starting national processing loop...


Progress: 1/36
 Greater_London — already done (1,983 grid cells)

Progress: 2/36
 Kent — already done (2,163 grid cells)

Progress: 3/36
 Surrey — already done (1,313 grid cells)

Progress: 4/36
 Hampshire — already done (1,817 grid cells)

Progress: 5/36
 Sussex — already done (1,020 grid cells)

Progress: 6/36
 Berkshire — already done (726 grid cells)

Progress: 7/36
 Oxfordshire — already done (882 grid cells)

Progress: 8/36
 Devon — already done (1,773 grid cells)

Progress: 9/36
 Cornwall — already done (1,245 grid cells)

Progress: 10/36
 Somerset — already done (169 grid cells)

Progress: 11/36
 Gloucestershire — already done (1,198 grid cells)

Progress: 12/36
 Wiltshire — already done (1,060 grid cells)

Progress: 13/36
 Dorset — already done (876 grid cells)

Progress: 14/36
 East_of_England — already done (7,341 grid cells)

Progress: 15/36
 East_Midlands — already done (2,281 grid cells)

Progress: 16/36
 West_Midlands — already done

In [11]:
#Merging into national grid
import glob

print("Merging all regional grid files...")
region_files = glob.glob(REGIONS_DIR + "region_*.csv")
print(f"Found {len(region_files)} region files")

frames = []
for f in sorted(region_files):
    df = pd.read_csv(f)
    frames.append(df)
    print(f"  {f.split('/')[-1]}: {len(df):,} grid cells")

national_grid = pd.concat(frames, ignore_index=True)

# Drop duplicates — border areas may appear in multiple regions
before = len(national_grid)
national_grid = national_grid.drop_duplicates(subset='grid_id', keep='first')
after  = len(national_grid)
print(f"\nDuplicate grid cells removed: {before - after:,}")
print(f"Final national grid: {after:,} unique grid cells")

# Re-normalise scores nationally
scaler = MinMaxScaler()
national_grid['hist_freq_score']     = scaler.fit_transform(national_grid[['log_freq']])
national_grid['hist_severity_score'] = scaler.fit_transform(national_grid[['serious_rate']])
national_grid['hist_speed_score']    = scaler.fit_transform(national_grid[['avg_speed_limit']])
national_grid['base_risk_score']     = (
    0.60 * national_grid['hist_freq_score'] +
    0.30 * national_grid['hist_severity_score'] +
    0.10 * national_grid['hist_speed_score']
)

print(f"\nRisk score distribution (national):")
print(national_grid['base_risk_score'].describe())

print(f"\nGrid cells per region:")
print(national_grid['region_name'].value_counts().to_string())

out = NATIONAL_DIR + "national_grid_history.csv"
national_grid.to_csv(out, index=False)
print(f"\n Saved national_grid_history.csv ({len(national_grid):,} grid cells)")

Merging all regional grid files...
Found 37 region files
  region_01_Greater_London.csv: 1,983 grid cells
  region_02a_Kent.csv: 2,163 grid cells
  region_02b_Surrey.csv: 1,313 grid cells
  region_02c_Hampshire.csv: 1,817 grid cells
  region_02d_Sussex.csv: 1,020 grid cells
  region_02e_Berkshire.csv: 726 grid cells
  region_02f_Oxfordshire.csv: 882 grid cells
  region_03a_Devon.csv: 1,773 grid cells
  region_03b_Cornwall.csv: 1,245 grid cells
  region_03c_Somerset.csv: 169 grid cells
  region_03d_Gloucestershire.csv: 1,198 grid cells
  region_03e_Wiltshire.csv: 1,060 grid cells
  region_03f_Dorset.csv: 876 grid cells
  region_04_East_of_England.csv: 7,341 grid cells
  region_05_East_Midlands.csv: 2,281 grid cells
  region_06_West_Midlands.csv: 1,133 grid cells
  region_07_Yorkshire_and_Humber.csv: 5,140 grid cells
  region_08a_Lancashire.csv: 1,511 grid cells
  region_08b_Cumbria.csv: 1,009 grid cells
  region_08c_Cheshire.csv: 1,296 grid cells
  region_09_North_East_England.csv: 1,73

In [12]:
import glob

print("Merging all regional grid files...")
region_files = glob.glob(REGIONS_DIR + "region_*.csv")
print(f"Found {len(region_files)} region files")

frames = []
for f in sorted(region_files):
    df = pd.read_csv(f)
    frames.append(df)
    print(f"  {f.split('/')[-1]}: {len(df):,} grid cells")

national_grid = pd.concat(frames, ignore_index=True)

# Dropping duplicates, border areas may appear in multiple regions
before = len(national_grid)
national_grid = national_grid.drop_duplicates(subset='grid_id', keep='first')
after  = len(national_grid)
print(f"\nDuplicate grid cells removed: {before - after:,}")
print(f"Final national grid: {after:,} unique grid cells")

# Re-normalise scores nationally
scaler = MinMaxScaler()
national_grid['hist_freq_score']     = scaler.fit_transform(national_grid[['log_freq']])
national_grid['hist_severity_score'] = scaler.fit_transform(national_grid[['serious_rate']])
national_grid['hist_speed_score']    = scaler.fit_transform(national_grid[['avg_speed_limit']])
national_grid['base_risk_score']     = (
    0.60 * national_grid['hist_freq_score'] +
    0.30 * national_grid['hist_severity_score'] +
    0.10 * national_grid['hist_speed_score']
)

print(f"\nRisk score distribution (national):")
print(national_grid['base_risk_score'].describe())

print(f"\nGrid cells per region:")
print(national_grid['region_name'].value_counts())

out = NATIONAL_DIR + "national_grid_history.csv"
national_grid.to_csv(out, index=False)
print(f"\n Saved national_grid_history.csv ({len(national_grid):,} grid cells)")

Merging all regional grid files...
Found 37 region files
  region_01_Greater_London.csv: 1,983 grid cells
  region_02a_Kent.csv: 2,163 grid cells
  region_02b_Surrey.csv: 1,313 grid cells
  region_02c_Hampshire.csv: 1,817 grid cells
  region_02d_Sussex.csv: 1,020 grid cells
  region_02e_Berkshire.csv: 726 grid cells
  region_02f_Oxfordshire.csv: 882 grid cells
  region_03a_Devon.csv: 1,773 grid cells
  region_03b_Cornwall.csv: 1,245 grid cells
  region_03c_Somerset.csv: 169 grid cells
  region_03d_Gloucestershire.csv: 1,198 grid cells
  region_03e_Wiltshire.csv: 1,060 grid cells
  region_03f_Dorset.csv: 876 grid cells
  region_04_East_of_England.csv: 7,341 grid cells
  region_05_East_Midlands.csv: 2,281 grid cells
  region_06_West_Midlands.csv: 1,133 grid cells
  region_07_Yorkshire_and_Humber.csv: 5,140 grid cells
  region_08a_Lancashire.csv: 1,511 grid cells
  region_08b_Cumbria.csv: 1,009 grid cells
  region_08c_Cheshire.csv: 1,296 grid cells
  region_09_North_East_England.csv: 1,73